# Prepare Data for RDR

The original dataset: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Setup

In [1]:
import numpy as np
import pandas as pd
import json

## Load Data

In [ ]:
df = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [3]:
df.shape

(4803, 20)

## EDA
### Remove Columns

In [4]:
cols = df.columns
cols

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

Some columns are not important for simple movie recommendations because they are either identifiers, redundant information, unstructured text, or features that add unnecessary complexity for rule-based recommendation.

The following columns are removed:

- `homepage` - website link, not useful for recommendation
- `id` - unique identifier only
- `keywords` - unstructured text, harder for simple rule creation
- `original_title` - redundant since `title` is already available
- `overview` - long text description, not needed for basic rules
- `production_companies` - too specific and less useful for general recommendation
- `production_countries` - often redundant with language preference
- `spoken_languages` - overlaps with `original_language` and can be messy
- `status` - usually values like Released, not helpful
- `tagline` - short promotional text, not useful for recommendation

In [5]:
cols_to_remove = ['homepage', 'id', 'keywords', 'original_title', 'overview',
                'production_companies', 'production_countries', 'spoken_languages', 
                'status', 'tagline']

# romove those columns
df = df.drop(cols_to_remove, axis=1)
df.head()

,budget,genres,original_language,popularity,release_date,revenue,runtime,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,150.437577,2009-12-10,2787965087,162.0,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",en,139.082615,2007-05-19,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,107.376788,2015-10-26,880674609,148.0,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",en,112.312950,2012-07-16,1084939099,165.0,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,43.926995,2012-03-07,284139100,132.0,John Carter,6.1,2124


In [6]:
df.shape

(4803, 10)

### Remake Columns

Some columns are useful for recommendation but arrive in a format that does not match simple rule predicates.

The following columns are transformed:

- `genres` - replace JSON id/name objects with a list of genre names only.
- `release_date` - parse to calendar year in `release_year`, then drop `release_date`; exact release day is omitted as unnecessary for basic rules.

In [7]:
df["genres"] = df["genres"].map(
    lambda v: [g["name"] for g in json.loads(str(v).strip())]
    if pd.notna(v) and str(v).strip()
    else []
)

In [8]:
df["release_year"] = (
    pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")
)
df = df.drop(columns=["release_date"])

In [9]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2787965087,162.0,Avatar,7.2,11800,2009
1,300000000,"[Adventure, Fantasy, Action]",en,139.082615,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007
2,245000000,"[Action, Adventure, Crime]",en,107.376788,880674609,148.0,Spectre,6.3,4466,2015
3,250000000,"[Action, Crime, Drama, Thriller]",en,112.312950,1084939099,165.0,The Dark Knight Rises,7.6,9106,2012
4,260000000,"[Action, Adventure, Science Fiction]",en,43.926995,284139100,132.0,John Carter,6.1,2124,2012


### Create Columns

Our RDR is **classification**, so the target should be a **discrete class**, not a continuous score. Predicting raw `vote_average` would be regression; binning turns the problem into ordered categories that rules can mention directly.

In [10]:
# quick range check
print("min:", df["vote_average"].min())
print("max:", df["vote_average"].max())

min: 0.0
max: 10.0


In [11]:
# create rating class with bins
df["rating_class"] = pd.cut(
    df["vote_average"],
    bins=4,
    labels=["Poor", "Average", "Good", "Excellent"],
    include_lowest=True,
    right=False
)

In [12]:
# check created bins
bins = pd.cut(
    df["vote_average"],
    bins=4,
    include_lowest=True,
    right=False
)

print(bins.cat.categories)

IntervalIndex([[0.0, 2.5), [2.5, 5.0), [5.0, 7.5), [7.5, 10.01)], dtype='interval[float64, left]')


We can see the bins are created equally as expected:

|    `vote_average`   | `rating_class` |
|---------------------|----------------|
| x &lt; 2.5          | Poor           |
| 2.5 &le; x &lt; 5.0 | Average        |
| 5.0 &le; x &lt; 7.5 | Good           |
| x &ge; 7.5          | Excellent      |

In [13]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year,rating_class
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2787965087,162.0,Avatar,7.2,11800,2009,Good
1,300000000,"[Adventure, Fantasy, Action]",en,139.082615,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007,Good
2,245000000,"[Action, Adventure, Crime]",en,107.376788,880674609,148.0,Spectre,6.3,4466,2015,Good
3,250000000,"[Action, Crime, Drama, Thriller]",en,112.312950,1084939099,165.0,The Dark Knight Rises,7.6,9106,2012,Excellent
4,260000000,"[Action, Adventure, Science Fiction]",en,43.926995,284139100,132.0,John Carter,6.1,2124,2012,Good


## Preprocess Data
### Deal with NaNs

In [14]:
df.isnull().sum()

budget               0
genres               0
original_language    0
popularity           0
revenue              0
runtime              2
title                0
vote_average         0
vote_count           0
release_year         1
rating_class         0
dtype: int64

In [15]:
# get columns with NaNs
nan_cols = df.isnull().sum() > 0
nan_cols = nan_cols[nan_cols].index.tolist()
print(f"Columns with NaNs: {nan_cols}")
print()

# explore unique values for columns with NaNs
print(df['runtime'].unique())
print(df['release_year'].unique())
print()

# imputation
print(f"Mean of runtime: {df['runtime'].mean()}")  # runtime -> mean
print(f"Mode of release_year: {df['release_year'].mode().iloc[0]}")  # release_year -> mode (mean year not meaningful)

Columns with NaNs: ['runtime', 'release_year']

[162. 169. 148. 165. 132. 139. 100. 141. 153. 151. 154. 106. 149. 143.
 150. 136. 144. 140. 161. 113. 187. 194. 147. 131. 124. 127. 130. 108.
 104. 142. 125. 114. 103. 115. 137. 116. 122.  93.  98.  91. 158.  96.
 109. 152.  94. 126. 112. 123. 135. 118.  97. 119. 102. 120. 121. 166.
  99. 183. 175. 138. 157.  92. 101. 111.  89. 105. 107. 129.  88.  85.
 163. 133.  95.  90. 110. 128. 156. 117. 146.  82.  78. 134. 170.  76.
 178.  84. 155. 145. 180. 172. 167. 201. 179.  83.  87.  80.  74.  81.
 177.  86. 164. 159. 191. 189. 214.   0.  75. 192. 160. 219. 248. 188.
 202. 173. 195.  77.  79.  63. 229. 193. 254.  72.  68. 199. 181. 242.
 338.  73. 216. 276.  nan 200.  69. 197. 184. 176. 185. 171. 174.  46.
  53.  42. 240.  41.  67. 238. 186.  70.  14. 225. 207.  66.  59.  25.
  47.  64.  60.]
<IntegerArray>
[2009, 2007, 2015, 2012, 2010, 2016, 2006, 2008, 2013, 2011, 2014, 2005, 1997,
 2004, 1999, 1995, 2003, 2001, 2002, 1998, 2000, 1990, 1991,

In [16]:
# fill NaNs
df["runtime"] = df["runtime"].fillna(df["runtime"].mean())
df["release_year"] = df["release_year"].fillna(df["release_year"].mode().iloc[0])

In [17]:
df.isnull().sum()

budget               0
genres               0
original_language    0
popularity           0
revenue              0
runtime              0
title                0
vote_average         0
vote_count           0
release_year         0
rating_class         0
dtype: int64

In [18]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year,rating_class
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2787965087,162.0,Avatar,7.2,11800,2009,Good
1,300000000,"[Adventure, Fantasy, Action]",en,139.082615,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007,Good
2,245000000,"[Action, Adventure, Crime]",en,107.376788,880674609,148.0,Spectre,6.3,4466,2015,Good
3,250000000,"[Action, Crime, Drama, Thriller]",en,112.312950,1084939099,165.0,The Dark Knight Rises,7.6,9106,2012,Excellent
4,260000000,"[Action, Adventure, Science Fiction]",en,43.926995,284139100,132.0,John Carter,6.1,2124,2012,Good


In [19]:
df.shape

(4803, 11)

## Save Data

In [ ]:
df.to_csv('../data/cleaned/movies.csv', index=False)